# Fine Tuning LLama using QLoRA

In this notebook, we fine-tune the **LLaMA 3.1 8B Instruct** model using **QLoRA** (Quantized Low-Rank Adaptation) on the **CaseHOLD** dataset from the `LexGLUE` benchmark.

**Note:** This notebook may take a significant amount of time to run for the first time, as it downloads the required model weights and dependencies. Ensure that you have sufficient GPU resources for optimal performance.
 
This notebook requires **20GB of VRAM** to run successfully.

### Task Overview

**CaseHOLD** is a legal reasoning task that requires the model to complete a legal case context by selecting the correct legal holding from a set of five multiple-choice options.

For more details on **CaseHOLD** visit: https://github.com/coastalcph/lex-glue

### What This Notebook Demonstrates

- **Efficient fine-tuning** using **4-bit NF4 quantization**
- **Parameter-efficient adaptation** with **LoRA**
- **Full training and evaluation pipeline**
- **Performance analysis and error breakdown**
- **Inference on the test set** with answer extraction and accuracy metrics

# Pre-requisites

To support features of this notebook with CoreAI, we need to install some libraries that are not pre-installed but are required for this notebook. 

## Create and Activate the Virtual Environment:
Open your terminal or command prompt within the Jupyter notebook. Navigate via `File -> New -> Terminal`.
Type `bash` to access a shell compatible with the following commands.
Navigate to the project directory where you want to set up the environment (where this notebook is located):

```bash
export PROJECT_NAME="Fine_Tuning_LLama"
export PIP_CACHE_DIR=`pwd`/.cache/pip
mkdir -p $PIP_CACHE_DIR
python -m venv --system-site-packages myvenv
source myvenv/bin/activate
pip install ipykernel
python -m ipykernel install --user --name=${PROJECT_NAME}_myvenv --display-name="Python (${PROJECT_NAME}_myvenv)"
echo ""; echo "Before continuing load the created Python kernel: Python (${PROJECT_NAME}_myvenv)"
```

Load the Python kernel described above before running the cell below (it might take a few seconds for the kernel to appear in the list of kernels).

The following will set the folder location for download so that they are local to the running container, to provide cache.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["ANONYMIZED_TELEMETRY"] = 'False'
def set_env_with_cache_dir(env_var_name: str, subdir: str):
    base_cache = os.path.join(os.getcwd(), ".cache")
    full_path = os.path.join(base_cache, subdir)
    os.environ[env_var_name] = full_path
    os.makedirs(full_path, exist_ok=True)
    print(f"{env_var_name}={full_path}")

set_env_with_cache_dir("PIP_CACHE_DIR", "pip")
set_env_with_cache_dir("HF_HOME", "huggingface")
set_env_with_cache_dir("TORCH_HOME", "torch")

## Install Required Libraries:

The rest of this notebook relies on the proper kernel to be loaded and environment variables to be set. 

In [ ]:
!. ./myvenv/bin/activate; pip install -r requirements.txt

For downloading the LLama 3.1 8B Instruct model you must have a hugging face token (HF_TOKEN) with 'READ' permission

Follow the below steps to generate and use a Hugging Face access token in your local environment or notebook.

### Step 1: Log in to Hugging Face

Go to [Hugging Face Login](https://huggingface.co/login ) and log in with your account.  
If you don't have an account, create one at: [Join Hugging Face](https://huggingface.co/join )

### Step 2: Go to Access Tokens

After logging in:
1. Click on your profile icon (top-right)
2. Select **Settings**
3. Click on **Access Tokens** in the left sidebar

### Step 3: Generate a New Token

1. Click **New token**
2. Give it a name (e.g., `jupyter-token`)
3. Choose the type as `READ`
4. Click **Generate**

**Copy the token now** — you won’t be able to see it again!

### Step 4: Use the Token in Your Notebook

fill the `HF_TOKEN` in the input of the below cell

In [ ]:
import os
os.environ["HF_TOKEN"] = input("Enter your HF Token")

The below cell sets up the foundational components for fine-tuning a large language model (LLM), particularly focusing on efficient training using tools like LoRA , bitsandbytes , and accelerate . It also includes logging configuration to track training progress.

In [ ]:
import torch
import random
from torch.utils.data import DataLoader,Subset
from datasets import load_dataset
from tqdm import tqdm
import json
import numpy as np
from peft import LoraConfig, get_peft_model, TaskType
import bitsandbytes as bnb
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from accelerate import Accelerator
from scipy import stats
import math
from torch.optim.lr_scheduler import LinearLR
import logging
logging.basicConfig(
    filename='training_log.txt',
    filemode='w',
    format='%(asctime)s - %(levelname)s - %(message)s',
    level=logging.INFO
)
console = logging.StreamHandler()
console.setLevel(logging.INFO)
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
console.setFormatter(formatter)
logging.getLogger('').addHandler(console)

This function is a simple utility to load the CaseHold dataset from the LEX-GLUE benchmark

In [ ]:
def load_casehold_data():
    dataset = load_dataset("lex_glue", "case_hold")
    return dataset
dataset = load_casehold_data()

This function takes a single example from the CaseHold dataset and formats it into a structured prompt suitable for instruction-tuning or evaluation of a language model. It presents the legal case context and multiple choices, then includes the correct answer in a format similar to how a model might generate it.

In [ ]:
def format_casehold_example(example):
    context = example["context"]
    endings = example["endings"]
    label = example["label"]
    
    choices_text = "\nChoices:"
    for i, ending in enumerate(endings):
        choices_text += f"\n{chr(65+i)}) {ending}"
    
    correct_answer = chr(65 + label)
    correct_ending = endings[label]
    
    formatted_text = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a legal expert specializing in case law analysis and legal holdings.<|eot_id|><|start_header_id|>user<|end_header_id|>

Read the following legal case context and choose the correct completion for the legal holding:

Legal Case Context:
{context}

Choose the correct completion:
{choices_text}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The correct answer is {correct_answer}: {correct_ending}<|eot_id|>"""
    
    return {"text": formatted_text}

This function sets up a pretrained causal language model (Llama-3.1-8B-Instruct) and its corresponding tokenizer , with 4-bit quantization using bitsandbytes for efficient inference or fine-tuning.

In [ ]:
def setup_model_and_tokenizer():
    model_name = "meta-llama/Llama-3.1-8B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",  
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,  
        bnb_4bit_quant_storage=torch.uint8,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    return model, tokenizer

This function defines and returns a LoRA (Low-Rank Adaptation) configuration for fine-tuning a causal language model , such as Llama or Mistral. LoRA is a parameter-efficient fine-tuning (PEFT) technique that significantly reduces memory usage and training cost by only updating low-rank matrices during training.

This is often used in combination with quantized models (like QLoRA), where the base model weights are frozen and only the LoRA adapters are trained.

In [ ]:
def setup_qlora_config():
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16,  
        lora_alpha=32,  
        lora_dropout=0.1, 
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        bias="none",
        use_rslora=False,
        init_lora_weights=True,
    )
    return lora_config

## Setting the fine tuning parameters

Please fill out the below parameters for fine Tuning the model

**Recommendation:** For quick testing, set `num_epochs_input = 3` by entering `3` in the input below.

In [ ]:
num_epochs_input = input("Enter the number of Epochs (any value between 3 to 6 is fine for this task): ")
try:
    NUM_EPOCHS = int(num_epochs_input)
    
    if NUM_EPOCHS < 3:
        print("Too few epochs. Defaulting to 3.")
        NUM_EPOCHS = 3
    elif NUM_EPOCHS > 6:
        print("More than 6 epochs not typically needed here. Defaulting to 6.")
        NUM_EPOCHS = 6
    else:
        print(f"Using {NUM_EPOCHS} epochs.")
        
except ValueError:
    print("Invalid input. Defaulting to 3 epochs.")
    NUM_EPOCHS = 3

**NOTE:** Setting a maximum number of steps per epoch can help speed up training by using only a subset of the data in each epoch. However, this may reduce model performance since the model won’t be exposed to the full dataset. When using this option, ensure that the `max_steps_per_epoch` value does not exceed the total number of training steps (`num_training_steps`) and is greater than the number of warmup steps (`warmup_steps`).

**Recommendation:** For quick testing, set `max_steps_input = 200` by entering `200` in the input below.

In [ ]:
max_steps_input = input("Enter Max steps per epoch to skip the full training (leave blank or enter 0 to run all steps): ")
try:
    MAX_STEPS_PER_EPOCH = int(max_steps_input)
    if MAX_STEPS_PER_EPOCH < 0:
        print("Negative steps not allowed. Defaulting to no limit.")
        MAX_STEPS_PER_EPOCH = None
    elif MAX_STEPS_PER_EPOCH == 0:
        print("No step limit set. Will run all steps.")
    else:
        print(f"Will run at most {MAX_STEPS_PER_EPOCH} steps per epoch (limited data exposure).")
        
except ValueError:
    print("Invalid input. Defaulting to no limit.")
    MAX_STEPS_PER_EPOCH = None

**NOTE:** You can choose how many examples you want to use for validation.
Using fewer examples will make validation faster, but the result might not represent the whole dataset well.
Enter a number that is not greater than the total validation examples.

**Recommendation:** For quick testing, set `total_val_examples  = 10` by entering `10` in the input below.

In [ ]:
total_val_examples = len(dataset['validation'])
val_examples_input = input(f"Enter the number of validation examples to use (max {total_val_examples}, 0 or blank for all): ")

try:
    VAL_EXAMPLES = int(val_examples_input)

    if VAL_EXAMPLES < 0 or VAL_EXAMPLES > total_val_examples:
        print(f"Invalid input. Using all {total_val_examples} examples for validation.")
        VAL_EXAMPLES = total_val_examples
    else:
        print(f"Will use {VAL_EXAMPLES} examples for validation.")

except ValueError:
    print(f"Invalid input. Defaulting to all {total_val_examples} examples.")
    VAL_EXAMPLES = total_val_examples

## Required Model Access

The **LLaMA 3.1 8B - Instruct** model is hosted on Hugging Face and is a **gated** model. To use it, you must first request access from Meta through Hugging Face:

1. [meta-llama/Llama-3-8B-Instruct](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct)

> **Access Instructions:**  
> - Log in to your Hugging Face account.  
> - Visit the link above.  
> - Click on the “Access repository” button and fill out the access request form, agreeing to Meta’s terms and conditions.

Once access is granted, there is no additional login step required if the Hugging Face token is correctly set as an environment variable.

## Training

This function fine-tunes a large language model using QLoRA and Hugging Face’s `Accelerator`. It handles the complete training workflow, including:

- Initializing the accelerator with mixed precision and gradient accumulation
- Formatting and tokenizing the dataset
- Preparing the model with QLoRA (parameter-efficient fine-tuning)
- Setting up data loaders, optimizer, and scheduler
- Running the training loop with validation, checkpointing, and early stopping
- Logging training statistics and saving the best and final models

It is designed for efficient training with support for mixed precision, modular checkpointing, and performance tracking.

**NOTE**: Depending on your GPU hardware, you can try setting different hyperparameters available in the training code to optimize performance.

In [ ]:
def train_model():
    accelerator = Accelerator(gradient_accumulation_steps=4, mixed_precision="fp16", project_dir="./logs")
    
    logging.info("Formatting dataset...")
    formatted_dataset = dataset.map(
        format_casehold_example,
        remove_columns=dataset["train"].column_names,
        desc="Formatting examples"
    )
    
    model, tokenizer = setup_model_and_tokenizer()
   
    logging.info("Applying enhanced QLoRA...")
    lora_config = setup_qlora_config()
    model = get_peft_model(model, lora_config)
    
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    all_params = sum(p.numel() for p in model.parameters())
    logging.info(f"Trainable parameters: {trainable_params:,}")
    logging.info(f"All parameters: {all_params:,}")
    logging.info(f"Percentage of trainable parameters: {100 * trainable_params / all_params:.2f}%")
    
    def tokenize_function(examples):
        tokenized = tokenizer(
            examples["text"],
            truncation=True,
            padding="max_length",
            max_length=1024,
            return_tensors=None  
        )
        return tokenized

    tokenized_dataset = formatted_dataset.map(
        tokenize_function, 
        batched=True,
        desc="Tokenizing",
        remove_columns=formatted_dataset["train"].column_names,
        load_from_cache_file=False  
    )
    
    def robust_data_collator(batch):
        input_ids_list = []
        attention_mask_list = []
        for item in batch:
            input_ids = torch.tensor(item["input_ids"], dtype=torch.long)
            attention_mask = torch.tensor(item["attention_mask"], dtype=torch.long)
            input_ids_list.append(input_ids)
            attention_mask_list.append(attention_mask)
        return {
            "input_ids": torch.stack(input_ids_list),
            "attention_mask": torch.stack(attention_mask_list),
        }
    
    train_dataloader = DataLoader(
        tokenized_dataset["train"], 
        batch_size=1, 
        shuffle=True,
        collate_fn=robust_data_collator,
        num_workers=1,  
        pin_memory=False 
    )

    indices = list(range(0, VAL_EXAMPLES))
    subset = Subset(tokenized_dataset["validation"], indices)
    validation_dataloader = DataLoader(
        subset,
        batch_size=1, 
        shuffle=False,  
        collate_fn=robust_data_collator,
        num_workers=1,  
        pin_memory=False 
    )

    logging.info(f"Training samples: {len(tokenized_dataset['train'])}")
    logging.info(f"Validation samples: {len(tokenized_dataset['validation'])}")

    num_epochs = NUM_EPOCHS
    gradient_accumulation_steps = 4
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0.01, betas=(0.9, 0.999), eps=1e-8)
    num_training_steps = len(train_dataloader) * NUM_EPOCHS / gradient_accumulation_steps   
    warmup_steps = min(100, num_training_steps // 10)
    scheduler = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_steps)
    
    model, optimizer, train_dataloader, validation_dataloader, scheduler = accelerator.prepare(model, optimizer, train_dataloader, validation_dataloader, scheduler)
    
    max_steps_per_epoch = MAX_STEPS_PER_EPOCH
    output_dir = "./casehold-llama-qlora"
    best_model_dir = os.path.join(output_dir, "best_model")
    checkpoint_dir = os.path.join(output_dir, "checkpoints")
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(best_model_dir, exist_ok=True)
    os.makedirs(checkpoint_dir, exist_ok=True)

    save_every_n_steps = 50  
    save_every_n_epochs = 1  
    validate_every_n_steps = 200
    patience_steps = 100     
    min_improvement = 0.001 
    
    logging.info(f"Starting enhanced training for {num_epochs} epochs...")
    logging.info(f"Total training steps: {num_training_steps}")
    logging.info(f"Warmup steps: {warmup_steps}")
    logging.info(f"Validation will run every {validate_every_n_steps} steps")
    logging.info(f"Checkpoints will be saved every {save_every_n_steps} steps")
    logging.info(f"Early stopping based on validation loss with patience: {patience_steps}")
    logging.info(f"Best model will be saved when validation loss improves by at least {min_improvement}")
    
    def run_validation(model, validation_dataloader):
        model.eval()
        validation_losses = []
        
        with torch.no_grad():
            val_progress = tqdm(validation_dataloader, desc="Validation", leave=False, disable=not accelerator.is_local_main_process)
            for batch in val_progress:
                input_ids = batch["input_ids"].to(model.device)
                attention_mask = batch["attention_mask"].to(model.device)
                
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
                val_loss = outputs.loss
                validation_losses.append(val_loss.item())
                
                val_progress.set_postfix({"val_loss": f"{val_loss.item():.4f}"})
        
        avg_val_loss = np.mean(validation_losses)
        std_val_loss = np.std(validation_losses)
        
        model.train()  
        return avg_val_loss, std_val_loss, validation_losses
        
    def save_model_checkpoint(model, tokenizer, output_path, config_data, is_best=False):
        accelerator.wait_for_everyone()
        accelerator.save_model(model, output_path)
        
        if accelerator.is_main_process:
            tokenizer.save_pretrained(output_path)
            
            config_data["saved_at"] = torch.distributed.get_rank() if torch.distributed.is_initialized() else 0
            config_data["is_best_model"] = is_best
            
            with open(os.path.join(output_path, "training_config.json"), 'w') as f:
                json.dump(config_data, f, indent=2)
            
            checkpoint_type = "BEST MODEL" if is_best else "CHECKPOINT"
            logging.info(f" {checkpoint_type} saved to {output_path}")
            
    
    model.train()
    global_step = 0
    best_loss = float('inf')
    steps_without_improvement = 0
    loss_history = []
    val_loss_history = []

    logging.info("Running initial validation...")
    initial_val_loss, initial_val_std, _ = run_validation(model, validation_dataloader)
    val_loss_history.append(initial_val_loss)
    best_val_loss = initial_val_loss
    logging.info(f"Initial validation loss: {initial_val_loss:.4f} (±{initial_val_std:.4f})")
    
    base_config = {
        "model_name": "meta-llama/Llama-3.1-8B-Instruct",
        "lora_config": {
            "task_type": str(lora_config.task_type),
            "r": lora_config.r,
            "lora_alpha": lora_config.lora_alpha,
            "lora_dropout": lora_config.lora_dropout,
            "target_modules": list(lora_config.target_modules),
            "bias": lora_config.bias,
            "use_rslora": lora_config.use_rslora,
            "init_lora_weights": lora_config.init_lora_weights,
        },
        "training_params": {
            "epochs": num_epochs,
            "learning_rate": 2e-4,
            "batch_size": 1,
            "gradient_accumulation_steps": gradient_accumulation_steps,
            "max_length": 1024,
            "max_steps_per_epoch": max_steps_per_epoch,
        }
    }

    for epoch in range(num_epochs):
        epoch_losses = []
        
        progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}", disable=not accelerator.is_local_main_process)
        for batch_idx, batch in enumerate(progress_bar):
            with accelerator.accumulate(model):
                input_ids = batch["input_ids"].to(model.device)
                attention_mask = batch["attention_mask"].to(model.device)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
                loss = outputs.loss
                current_loss = loss.item()
                epoch_losses.append(current_loss)
                loss_history.append(current_loss)

                accelerator.backward(loss)
                
                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                if global_step < warmup_steps:
                    scheduler.step()
                optimizer.zero_grad()
                global_step += 1
                avg_loss = np.mean(epoch_losses)
                progress_bar.set_postfix({
                    "loss": f"{current_loss:.4f}",
                    "avg_loss": f"{avg_loss:.4f}",
                    "best_training_loss": f"{best_loss:.4f}",
                    "best_val": f"{best_val_loss:.4f}",
                    "lr": f"{optimizer.param_groups[0]['lr']:.2e}"
                })
                
                if global_step % validate_every_n_steps == 0:
                    logging.info(f"\nRunning validation at step {global_step}...")
                    val_loss, val_std, _ = run_validation(model, validation_dataloader)
                    val_loss_history.append(val_loss)
                    
                    logging.info(f"Validation loss: {val_loss:.4f} (±{val_std:.4f})")
                    logging.info(f"Best validation loss: {best_val_loss:.4f}")
                    
                    if val_loss < (best_val_loss - min_improvement):
                        old_best_val = best_val_loss
                        best_val_loss = val_loss
                        steps_without_improvement = 0
                        
                        best_config = base_config.copy()
                        best_config.update({
                            "best_validation_loss": float(best_val_loss),
                            "validation_improvement": float(old_best_val - best_val_loss),
                            "validation_std": float(val_std),
                            "step_achieved": global_step,
                            "epoch_achieved": epoch + 1,
                            "total_steps": global_step,
                            "current_train_loss": current_loss
                        })
                        
                        save_model_checkpoint(model, tokenizer, best_model_dir, best_config, is_best=True)
                        logging.info(f" NEW BEST MODEL! Val Loss: {old_best_val:.4f} → {best_val_loss:.4f} (improvement: {old_best_val - best_val_loss:.4f})")
                        
                    else:
                        steps_without_improvement += validate_every_n_steps
                        logging.info(f"No validation improvement. Steps without improvement: {steps_without_improvement}")
                
                if global_step % save_every_n_steps == 0:
                    step_config = base_config.copy()
                    step_config.update({
                        "current_step": global_step,
                        "current_epoch": epoch + 1,
                        "current_train_loss": current_loss,
                        "average_train_loss": avg_loss,
                        "best_validation_loss": float(best_val_loss),
                        "steps_without_improvement": steps_without_improvement,
                        "total_steps": global_step
                    })
                    
                    checkpoint_path = os.path.join(checkpoint_dir, f"step_{global_step}")
                    save_model_checkpoint(model, tokenizer, checkpoint_path, step_config, is_best=False)
                
                if steps_without_improvement >= patience_steps:
                    logging.info(f"Early stopping triggered! No validation improvement for {patience_steps} steps.")
                    logging.info(f"Best validation loss achieved: {best_val_loss:.4f}")
                    break
                    
                if batch_idx >= max_steps_per_epoch:
                    break
                

        if steps_without_improvement >= patience_steps:
            break
            
        if (epoch + 1) % save_every_n_epochs == 0:
            epoch_avg_loss = np.mean(epoch_losses)
            epoch_std_loss = np.std(epoch_losses)


            logging.info(f"\nRunning end-of-epoch validation...")
            epoch_val_loss, epoch_val_std, _ = run_validation(model, validation_dataloader)
            val_loss_history.append(epoch_val_loss)
            
            epoch_config = base_config.copy()
            epoch_config.update({
                "completed_epoch": epoch + 1,
                "epoch_average_train_loss": float(epoch_avg_loss),
                "epoch_std_train_loss": float(epoch_std_loss),
                "epoch_validation_loss": float(epoch_val_loss),
                "epoch_validation_std": float(epoch_val_std),
                "best_validation_loss": float(best_val_loss),
                "total_steps": global_step,
                "best_loss_so_far": float(best_loss)
            })
            
            epoch_path = os.path.join(checkpoint_dir, f"epoch_{epoch + 1}")
            save_model_checkpoint(model, tokenizer, epoch_path, epoch_config, is_best=False)
        
        epoch_avg_loss = np.mean(epoch_losses)
        epoch_std_loss = np.std(epoch_losses)
        
        logging.info(f"Epoch {epoch+1} completed:")
        logging.info(f"  Average loss: {epoch_avg_loss:.4f}")
        logging.info(f"  Loss std: {epoch_std_loss:.4f}")
        logging.info(f"  Best train loss so far: {best_loss:.4f}")
        logging.info(f"  Steps without improvement: {steps_without_improvement}")
        
    final_config = base_config.copy()
    final_config.update({
        "final_train_loss": float(np.mean(loss_history[-10:]) if loss_history else 0),
        "final_validation_loss": float(val_loss_history[-1] if val_loss_history else 0),
        "best_validation_loss": float(best_val_loss),
        "total_steps": global_step,
        "completed_epochs": epoch + 1,
        "training_completed": True
    })
    
    final_path = os.path.join(output_dir, "final_model")
    save_model_checkpoint(model, tokenizer, final_path, final_config, is_best=False)
    if len(loss_history) > 1:
        x = np.arange(len(loss_history))
        train_slope, train_intercept, train_r_value, train_p_value, train_std_err = stats.linregress(x, loss_history)
        
        val_slope, val_intercept, val_r_value, val_p_value, val_std_err = 0, 0, 0, 0, 0
        if len(val_loss_history) > 1:
            val_x = np.arange(len(val_loss_history))
            val_slope, val_intercept, val_r_value, val_p_value, val_std_err = stats.linregress(val_x, val_loss_history)
        
        logging.info(f"\n" + "="*50)
        logging.info(f"TRAINING COMPLETED!")
        logging.info(f"\n" + "="*50)
        logging.info(f"Final Statistics:")
        logging.info(f"  Best validation loss: {best_val_loss:.4f}")
        logging.info(f"  Final train loss: {np.mean(loss_history[-50:]):.4f}")
        logging.info(f"  Final validation loss: {val_loss_history[-1] if val_loss_history else 'N/A':.4f}")
        logging.info(f"  Train loss trend slope: {train_slope:.6f}")
        logging.info(f"  Validation loss trend slope: {val_slope:.6f}")
        logging.info(f"  Train loss correlation: {train_r_value:.4f}")
        logging.info(f"  Validation loss correlation: {val_r_value:.4f}")
        logging.info(f"  Train loss reduction: {loss_history[0] - loss_history[-1]:.4f}")
        logging.info(f"  Total training steps: {global_step}")
        logging.info(f"  Total validation runs: {len(val_loss_history)}")
        logging.info(f"")
        logging.info(f"Saved Models:")
        logging.info(f"  Best model (validation): {best_model_dir}")
        logging.info(f"  Final model: {final_path}")
        logging.info(f"  Checkpoints: {checkpoint_dir}")
        logging.info(f"\n" + "="*50)
    
    return model, tokenizer

Run the below cell to start the training

In [ ]:
print("===CaseHOLD LLaMA 3.1 8B QLoRA Training ===\n")
print("\nStarting training process...")
model, tokenizer = train_model()

### Model Evaluation on Test Split

This function tests a trained language model on a selected number of examples from a test dataset. 

It randomly picks the specified number of samples from the test set and, for each sample:
- Creates a prompt that includes the legal case context and multiple-choice answers.
- Uses the model to generate a predicted answer choice.
- Compares the predicted answer to the correct one and records whether it was correct.

The function logs details for each test case and tracks overall accuracy as it goes. After testing all samples, it calculates and logs the final accuracy and error rate.

It also performs further analysis on the model’s performance by answer choice and errors, then returns all the results and analysis data.

In [ ]:
def test_trained_model(model, tokenizer, dataset, num_samples=50):
    logging.info("\n=== Model Testing on Test Split ===")
    test_split = dataset['test']
    
    if num_samples > len(test_split):
        num_samples = len(test_split)
        logging.info(f"Requested {num_samples} samples, but test set only has {len(test_split)}. Using all available.")
    
    indices = random.sample(range(len(test_split)), num_samples)
    
    logging.info(f"Testing on {num_samples} samples from test split...")
    
    results = []
    detailed_results = []
    
    for i, idx in enumerate(indices):
        example = test_split[idx]
        
        logging.info(f"\nTest Case {i+1}/{num_samples}:")
        logging.info(f"Context: {example['context'][:100]}...")
        
        choices = example['endings']
        
        choices_text = "\nChoices:"
        for j, choice in enumerate(choices):
            choices_text += f"\n{chr(65+j)}) {choice}"
        
        prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a legal expert specializing in case law analysis and legal holdings.<|eot_id|><|start_header_id|>user<|end_header_id|>
Read the following legal case context and choose the correct completion for the legal holding:
Legal Case Context:
{example['context']}
Choose the correct completion:
{choices_text}

Please respond with just the letter (A, B, C, D, or E) of the correct choice.<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                inputs.input_ids,
                max_new_tokens=5,
                do_sample=True,
                temperature=0.1,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.1
            )
        
        generated_tokens = outputs[0][inputs.input_ids.shape[-1]:]
        generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()       
        expected_letter = chr(65 + example['label'])  
        correct = (generated_text == expected_letter)
        results.append(correct)
        
        detailed_results.append({
            'test_case_id': i + 1,
            'dataset_index': idx,
            'context': example['context'],
            'choices': choices,
            'expected_label': example['label'],
            'expected_letter': expected_letter,
            'predicted_letter': generated_text,
            'correct': correct,
            'full_response': generated_text,
            'confidence_score': None 
        })
        
        logging.info(f"Expected: {expected_letter}")
        logging.info(f"Predicted: {generated_text}")
        logging.info(f"Correct: {correct}")
        
        if (i + 1) % 10 == 0:
            current_accuracy = np.mean(results)
            logging.info(f"Progress: {i+1}/{num_samples} - Current Accuracy: {current_accuracy:.2%}")
    
    accuracy = np.mean(results)
    correct_count = sum(results)
    total_count = len(results)
    
    logging.info(f"\n=== Final Test Results ===")
    logging.info(f"Total samples tested: {total_count}")
    logging.info(f"Correct predictions: {correct_count}")
    logging.info(f"Accuracy: {accuracy:.2%}")
    logging.info(f"Error rate: {(1-accuracy):.2%}")
    
    choice_performance = analyze_choice_performance(detailed_results)
    print_choice_analysis(choice_performance)
    
    error_analysis = analyze_errors(detailed_results)
    print_error_analysis(error_analysis)
    
    return {
        'results': results,
        'detailed_results': detailed_results,
        'accuracy': accuracy,
        'correct_count': correct_count,
        'total_count': total_count,
        'choice_performance': choice_performance,
        'error_analysis': error_analysis
    }

This function analyzes the model’s performance for each possible answer choice (A–E). It calculates how many times each choice was the correct answer, how often the model predicted it correctly, and how often it was predicted as something else. This helps in understanding if the model is biased toward or against certain choices and gives a breakdown of its performance per answer label.

In [ ]:
def analyze_choice_performance(detailed_results):
    choice_stats = {}
    for result in detailed_results:
        expected = result['expected_letter']
        predicted = result['predicted_letter']
        
        if expected not in choice_stats:
            choice_stats[expected] = {
                'total_expected': 0,
                'correct_predictions': 0,
                'predicted_as': {'A': 0, 'B': 0, 'C': 0, 'D': 0, 'E': 0, 'None': 0}
            }
        
        choice_stats[expected]['total_expected'] += 1
        if result['correct']:
            choice_stats[expected]['correct_predictions'] += 1
        
        pred_key = predicted if predicted else 'None'
        choice_stats[expected]['predicted_as'][pred_key] += 1
    
    return choice_stats

This function prints a detailed analysis of the model’s performance for each answer choice (A–E), based on the output from `analyze_choice_performance`.

For each choice, it logs:
- How many times it was the correct answer (`total_expected`)
- How many times the model predicted it correctly (`correct_predictions`)
- The accuracy for that choice
- A breakdown of what the model actually predicted when that choice was correct

This helps identify patterns or biases in the model’s predictions across different answer options.

In [ ]:
def print_choice_analysis(choice_performance):
    logging.info(f"\n=== Performance by Choice Position ===")
    
    for choice in ['A', 'B', 'C', 'D', 'E']:
        if choice in choice_performance:
            stats = choice_performance[choice]
            accuracy = stats['correct_predictions'] / stats['total_expected'] if stats['total_expected'] > 0 else 0
            
            logging.info(f"\nChoice {choice}:")
            logging.info(f"  Total expected: {stats['total_expected']}")
            logging.info(f"  Correctly predicted: {stats['correct_predictions']}")
            logging.info(f"  Accuracy: {accuracy:.2%}")
            logging.info(f"  Predicted as:")
            for pred_choice, count in stats['predicted_as'].items():
                if count > 0:
                    percentage = count / stats['total_expected'] * 100
                    logging.info(f"    {pred_choice}: {count} ({percentage:.1f}%)")

This function analyzes model prediction errors from a list of detailed result dictionaries. It identifies:

- Cases where the model made **no prediction**
- Cases where the model made the **wrong prediction**
- **Common confusions** between expected and predicted answer choices (e.g., "B→C")

Returns a dictionary summarizing error types and confusion patterns to help understand model weaknesses.


In [ ]:
def analyze_errors(detailed_results):
    errors = [r for r in detailed_results if not r['correct']]
    error_patterns = {
        'no_prediction': 0,
        'wrong_choice': 0,
        'common_confusions': {}
    }
    
    for error in errors:
        if error['predicted_letter'] is None:
            error_patterns['no_prediction'] += 1
        else:
            error_patterns['wrong_choice'] += 1
            
            expected = error['expected_letter']
            predicted = error['predicted_letter']
            confusion_key = f"{expected}→{predicted}"
            
            if confusion_key not in error_patterns['common_confusions']:
                error_patterns['common_confusions'][confusion_key] = 0
            error_patterns['common_confusions'][confusion_key] += 1
    
    return error_patterns

This function takes in the output of the error analysis and logs a summary. It shows how many times the model failed to make a prediction, how often it made the wrong prediction, and lists the most common confusion patterns (i.e., expected vs. predicted choices). It helps in understanding the types of errors the model is making.

In [ ]:
def print_error_analysis(error_analysis):
    logging.info(f"\n=== Error Analysis ===")
    logging.info(f"No prediction made: {error_analysis['no_prediction']}")
    logging.info(f"Wrong choice predicted: {error_analysis['wrong_choice']}")
    
    if error_analysis['common_confusions']:
        logging.info(f"\nMost common confusions:")
        sorted_confusions = sorted(
            error_analysis['common_confusions'].items(),
            key=lambda x: x[1],
            reverse=True
        )
        
        for confusion, count in sorted_confusions[:5]:
            logging.info(f"  {confusion}: {count} times")


In [ ]:
num_samples = input("Enter the number of examples on which you would like to do testing: ")

try:
    num_samples = int(num_samples)
    total_test_samples = len(dataset["test"])

    if num_samples <= 0:
        logging.info("Invalid input. Using all test examples.")
        num_samples = total_test_samples
    elif num_samples > total_test_samples:
        logging.info(f" Input exceeds available test examples ({total_test_samples}). Using all examples.")
        num_samples = total_test_samples
    else:
        logging.info(f" Using {num_samples} test examples.")
        
except ValueError:
    logging.info("Invalid input. Using all test examples.")
    num_samples = len(dataset["test"])

Finally we call the `test_trained_model` function to do model inference

In [ ]:
logging.info("="*60)
logging.info("COMPREHENSIVE MODEL EVALUATION")
logging.info("="*60)

test_results = test_trained_model( model, tokenizer, dataset=dataset, num_samples=num_samples)

logging.info(f"\n" + "="*60)
logging.info("EVALUATION SUMMARY")
logging.info(f"\n" + "="*60)
logging.info(f"Samples tested: {test_results['total_count']}")
logging.info(f"Model accuracy: {test_results['accuracy']:.2%}")